# 1. VLM 2:4 sparsity 변환 속도 측정

### 실험 환경

- **모델:** BLIP-2 (`blip2-opt-2.7b`) Vision Model
- **변환:** ONNX → TensorRT (FP16 + `SPARSE_WEIGHTS`)
- **적용 범위:** 전체 Linear 레이어 446개

### 방법

```
BLIP-2 로드 (FP16)
    └─ 전체 Linear 446개에 2:4 sparsity 적용 (WeightNormSparsifier)
        └─ Vision Model만 ONNX export
            └─ TensorRT 변환 (FP16 + SPARSE_WEIGHTS)
                └─ 100회 추론 측정
```

> Language Model은 autoregressive 구조로 TensorRT 변환이 복잡하여 Vision Model만 측정
>

In [9]:
from transformers import Blip2Config, Blip2ForConditionalGeneration
import torch
import time
import numpy as np

# config 명시적으로 설정
config = Blip2Config.from_pretrained("Salesforce/blip2-opt-2.7b")
model = Blip2ForConditionalGeneration(config)
model = model.cuda().half()
model.eval()
print(f"완료! 파라미터: {sum(p.numel() for p in model.parameters())/1e9:.2f}B")

KeyboardInterrupt: 

In [ ]:
import torch
import os
from torch.ao.pruning import WeightNormSparsifier

os.makedirs("blip2_onnx", exist_ok=True)
dummy_pixel = torch.randn(1, 3, 224, 224, dtype=torch.float16).cuda()

# 1. Dense 모델 ONNX 저장 (위에서 로드한 model 사용)
print("Dense ONNX 변환 중...")
with torch.no_grad():
    torch.onnx.export(
        model.vision_model,
        dummy_pixel,
        "blip2_onnx/blip2_vision_dense.onnx",
        input_names=["pixel_values"],
        output_names=["output"],
        opset_version=17,
        do_constant_folding=True
    )
print("Dense ONNX 저장 완료!")

# 2. Sparse 모델 별도 생성
print("Sparse 모델 생성 중...")
sparse_model = Blip2ForConditionalGeneration(config).cuda().half().eval()

sparse_config = []
for name, module in sparse_model.named_modules():
    if isinstance(module, torch.nn.Linear):
        sparse_config.append({"tensor_fqn": f"{name}.weight"})

sparsifier = WeightNormSparsifier(
    sparsity_level=1.0,
    sparse_block_shape=(1, 4),
    zeros_per_block=2
)
sparsifier.prepare(sparse_model, sparse_config)
sparsifier.step()
sparsifier.squash_mask()
print(f"Sparsity 적용 완료! {len(sparse_config)}개 레이어")

# 3. Sparse ONNX 저장
print("Sparse ONNX 변환 중...")
with torch.no_grad():
    torch.onnx.export(
        sparse_model.vision_model,
        dummy_pixel,
        "blip2_onnx/blip2_vision_sparse.onnx",
        input_names=["pixel_values"],
        output_names=["output"],
        opset_version=17,
        do_constant_folding=True
    )
print("Sparse ONNX 저장 완료!")

Dense ONNX 변환 중...


/tmp/ipykernel_163955/2722503413.py:11: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(
/home/hambugy/sunghyun/venv/lib/python3.10/site-packages/transformers/models/blip_2/modeling_blip_2.py:250: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to o

Dense ONNX 저장 완료!
Sparse 모델 생성 중...
Sparsity 적용 완료! 446개 레이어
Sparse ONNX 변환 중...


/tmp/ipykernel_163955/2722503413.py:44: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(


Sparse ONNX 저장 완료!


In [ ]:
import tensorrt as trt

logger = trt.Logger(trt.Logger.WARNING)

def build_engine(onnx_path, sparse=False):
    builder = trt.Builder(logger)
    network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
    parser = trt.OnnxParser(network, logger)
    config = builder.create_builder_config()
    
    config.set_flag(trt.BuilderFlag.FP16)
    if sparse:
        config.set_flag(trt.BuilderFlag.SPARSE_WEIGHTS)
        print("Sparse Weights 활성화!")
    
    with open(onnx_path, 'rb') as f:
        parser.parse(f.read())
    
    profile = builder.create_optimization_profile()
    profile.set_shape("pixel_values", (1,3,224,224), (1,3,224,224), (1,3,224,224))
    config.add_optimization_profile(profile)
    
    print(f"엔진 빌드 중... ({onnx_path})")
    engine = builder.build_serialized_network(network, config)
    print("완료!")
    return engine

# Dense 엔진 빌드 & 저장
engine_dense = build_engine("blip2_onnx/blip2_vision_dense.onnx", sparse=False)
with open("blip2_onnx/blip2_vision_dense.trt", "wb") as f:
    f.write(engine_dense)
print("Dense 엔진 저장 완료!")

# Sparse 엔진 빌드 & 저장
engine_sparse = build_engine("blip2_onnx/blip2_vision_sparse.onnx", sparse=True)
with open("blip2_onnx/blip2_vision_sparse.trt", "wb") as f:
    f.write(engine_sparse)
print("Sparse 엔진 저장 완료!")

엔진 빌드 중... (blip2_onnx/blip2_vision_dense.onnx)
완료!
Dense 엔진 저장 완료!
Sparse Weights 활성화!
엔진 빌드 중... (blip2_onnx/blip2_vision_sparse.onnx)
완료!
Sparse 엔진 저장 완료!


In [6]:
# output 이름이랑 shape 확인
runtime = trt.Runtime(logger)
with open("blip2_onnx/blip2_vision_dense.trt", "rb") as f:
    engine = runtime.deserialize_cuda_engine(f.read())

print(f"입출력 텐서 수: {engine.num_io_tensors}")
for i in range(engine.num_io_tensors):
    name = engine.get_tensor_name(i)
    shape = engine.get_tensor_shape(name)
    dtype = engine.get_tensor_dtype(name)
    mode = engine.get_tensor_mode(name)
    print(f"  {name}: shape={shape}, dtype={dtype}, mode={mode}")

입출력 텐서 수: 3
  pixel_values: shape=(1, 3, 224, 224), dtype=DataType.HALF, mode=TensorIOMode.INPUT
  output: shape=(1, 257, 1408), dtype=DataType.HALF, mode=TensorIOMode.OUTPUT
  2945: shape=(1, 1408), dtype=DataType.HALF, mode=TensorIOMode.OUTPUT


In [ ]:
import tensorrt as trt
import torch
import time
import numpy as np

logger = trt.Logger(trt.Logger.WARNING)

def run_engine(engine_path, input_data):
    runtime = trt.Runtime(logger)
    with open(engine_path, "rb") as f:
        engine = runtime.deserialize_cuda_engine(f.read())
    context = engine.create_execution_context()
    
    input_tensor = torch.from_numpy(input_data).cuda()
    output_tensor1 = torch.zeros((1, 257, 1408), dtype=torch.float16).cuda()
    output_tensor2 = torch.zeros((1, 1408), dtype=torch.float16).cuda()
    
    context.set_tensor_address("pixel_values", input_tensor.data_ptr())
    context.set_tensor_address("output", output_tensor1.data_ptr())
    context.set_tensor_address("2945", output_tensor2.data_ptr())
    
    stream = torch.cuda.Stream()
    
    # 워밍업
    for _ in range(10):
        context.execute_async_v3(stream.cuda_stream)
    torch.cuda.synchronize()
    
    # 측정
    times = []
    for _ in range(100):
        start = time.perf_counter()
        context.execute_async_v3(stream.cuda_stream)
        torch.cuda.synchronize()
        times.append((time.perf_counter() - start) * 1000)
    
    return np.mean(times), np.min(times)

dummy = np.random.randn(1, 3, 224, 224).astype(np.float16)

print("Dense 측정 중...")
dense_mean, dense_min = run_engine("blip2_onnx/blip2_vision_dense.trt", dummy)
print(f"Dense  평균: {dense_mean:.2f}ms, 최소: {dense_min:.2f}ms")

print("Sparse 측정 중...")
sparse_mean, sparse_min = run_engine("blip2_onnx/blip2_vision_sparse.trt", dummy)
print(f"Sparse 평균: {sparse_mean:.2f}ms, 최소: {sparse_min:.2f}ms")

print(f"\n속도 향상: {dense_mean/sparse_mean:.2f}x")

Dense 측정 중...
Dense  평균: 93.24ms, 최소: 92.98ms
Sparse 측정 중...
Sparse 평균: 81.67ms, 최소: 81.56ms

속도 향상: 1.14x
